<a href="https://colab.research.google.com/github/MhThorq/AnomaliEwallet/blob/main/Fraud_DetCompress.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload() # Unggah file kaggle.json di sini

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset langsung ke Colab
!kaggle competitions download -c ieee-fraud-detection
!unzip ieee-fraud-detection.zip

Saving kaggle.json to kaggle.json
 91% 108M/118M [00:00<00:00, 1.10GB/s]
100% 118M/118M [00:00<00:00, 806MB/s] 
Archive:  ieee-fraud-detection.zip
  inflating: sample_submission.csv   
  inflating: test_identity.csv       
  inflating: test_transaction.csv    
  inflating: train_identity.csv      
  inflating: train_transaction.csv   


In [ ]:
import numpy as np
import pandas as pd

def reduce_mem_usage(df):
    """ Fungsi yang diperbaiki untuk menghindari overflow pada data finansial """
    start_mem = df.memory_usage().sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                # Optimalisasi untuk tipe Integer tetap sama
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                # PERBAIKAN: Hindari penggunaan float16 untuk data finansial
                # Kita hanya akan menurunkan float64 ke float32 jika memungkinkan
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memori berkurang menjadi {end_mem:.2f} MB ({((start_mem - end_mem) / start_mem * 100):.1f}% berkurang)')
    return df

In [ ]:
import gc
# Load data
train_transaction = pd.read_csv('train_transaction.csv')
train_identity = pd.read_csv('train_identity.csv')

# Optimasi memori segera setelah load
train_transaction = reduce_mem_usage(train_transaction)
train_identity = reduce_mem_usage(train_identity)

# Gabungkan data (Merging)
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')

# Hapus variabel lama untuk mengosongkan RAM
del train_transaction, train_identity
gc.collect()

print(f"Data gabungan siap dengan bentuk: {train.shape}")

Memori berkurang menjadi 916.30 MB (48.4% berkurang)
Memori berkurang menjadi 31.91 MB (29.3% berkurang)
Data gabungan siap dengan bentuk: (590540, 434)


In [ ]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

# Gunakan opsi ini agar semua kolom terlihat saat di-display
pd.set_option('display.max_columns', 500)

In [ ]:
from google.colab import files
import os

# Nama file sementara yang akan disimpan di environment Colab
file_name = 'processed_data.pkl'

# 1. Simpan data 'train' ke penyimpanan lokal sementara Colab
# Variabel 'train' adalah dataframe hasil merging dan optimasi memori
train.to_pickle(file_name)

# 2. Trigger proses download ke browser lokal
files.download(file_name)

print(f"Proses download untuk {file_name} telah dimulai!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Proses download untuk processed_data.pkl telah dimulai!
